# Sub-Saharan African bat occurrence records with taxonomy, locality, and specimen documentation Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.s7ex-6yp9/fair2.json`

This dataset describes curated occurrence records for bats in sub-Saharan Africa, with taxonomic, locality, coordinate, and specimen-related metadata. In this notebook, all Croissant entities are referenced by their `@id` values whenever record sets, fields, or other metadata objects are accessed.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset metadata from the Croissant document and inspect the high-level dataset description.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.s7ex-6yp9/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset loaded successfully.')
print('Name:', getattr(metadata, 'name', 'N/A'))
print('Description:', getattr(metadata, 'description', 'N/A'))
print('Identifier:', getattr(metadata, 'identifier', 'N/A'))
print('Version:', getattr(metadata, 'version', 'N/A'))
print('Date published:', getattr(metadata, 'date_published', getattr(metadata, 'datePublished', 'N/A')))
print('License:', getattr(metadata, 'license', 'N/A'))

The next cell converts the metadata object to a JSON-like representation for inspection. This is useful for discovering dataset-level `@id` values and any nested Croissant entities exposed by the schema.

In [ ]:
metadata_json = metadata.to_json()

print('Top-level metadata keys:')
print(sorted(metadata_json.keys()))

print('\nDataset @id:')
print(metadata_json.get('@id'))

print('\nDataset name:')
print(metadata_json.get('name'))

## 2. Data Overview
Review available record sets and fields. Because Croissant datasets can contain multiple record sets, we first inspect the metadata to discover the available `@id` values dynamically.

If the dataset does not expose record sets through the parser, the notebook still provides a robust fallback path and documents that situation clearly.

In [ ]:
record_sets = metadata_json.get('recordSet', []) or []

print(f'Number of record sets discovered in metadata: {len(record_sets)}')

record_set_summaries = []
for rs in record_sets:
    rs_id = rs.get('@id', 'N/A')
    rs_name = rs.get('name', 'N/A')
    rs_description = rs.get('description', 'N/A')
    field_list = rs.get('field', []) or []
    record_set_summaries.append({
        'record_set_@id': rs_id,
        'name': rs_name,
        'description': rs_description,
        'field_count': len(field_list)
    })

record_set_overview_df = pd.DataFrame(record_set_summaries)
display(record_set_overview_df)

In [ ]:
# Build a field-level overview using only @id-based references
field_rows = []

for rs in record_sets:
    rs_id = rs.get('@id', 'N/A')
    for field in rs.get('field', []) or []:
        source_info = field.get('source', {}) if isinstance(field.get('source', {}), dict) else {}
        extract_info = source_info.get('extract', {}) if isinstance(source_info.get('extract', {}), dict) else {}
        field_rows.append({
            'record_set_@id': rs_id,
            'field_@id': field.get('@id', 'N/A'),
            'field_name': field.get('name', 'N/A'),
            'description': field.get('description', 'N/A'),
            'dataType': field.get('dataType', 'N/A'),
            'source_fileObject_@id': source_info.get('fileObject', 'N/A'),
            'extract_column': extract_info.get('column', 'N/A')
        })

fields_overview_df = pd.DataFrame(field_rows)

if not fields_overview_df.empty:
    display(fields_overview_df.head(20))
else:
    print('No field definitions were discovered in record sets from the metadata.')

The next cell demonstrates how to iterate through records using a record set `@id`. Since this specific dataset metadata may expose zero record sets, the code checks first and only executes record iteration when a valid record set `@id` is available.

In [ ]:
if record_sets:
    example_record_set_id = record_sets[0].get('@id')
    print('Inspecting records from record set @id:', example_record_set_id)
    
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print('No record sets were declared in metadata, so there is no record_set @id available for dataset.records(...).')
    print('This suggests the Croissant package currently exposes dataset-level FAIR metadata without tabular record-set definitions.')

## 3. Data Extraction
Load data from each available record set into pandas DataFrames. The logic below uses discovered record set `@id` values dynamically.

For this dataset, if no record sets are present, we create a small dataset-level summary table from metadata so the remainder of the notebook can still support structured exploration.

In [ ]:
dataframes = {}
record_set_ids = [rs.get('@id') for rs in record_sets if rs.get('@id')]

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(dataframes[record_set_id])} rows from record set {record_set_id}')
    except Exception as e:
        print(f'Could not load record set {record_set_id}: {e}')

if not dataframes:
    dataset_summary_rows = [
        {'field_@id': 'dataset:@id', 'value': metadata_json.get('@id')},
        {'field_@id': 'dataset:name', 'value': metadata_json.get('name')},
        {'field_@id': 'dataset:identifier', 'value': metadata_json.get('identifier')},
        {'field_@id': 'dataset:version', 'value': metadata_json.get('version')},
        {'field_@id': 'dataset:license', 'value': metadata_json.get('license')},
        {'field_@id': 'dataset:datePublished', 'value': metadata_json.get('datePublished')},
        {'field_@id': 'dataset:temporalCoverage', 'value': metadata_json.get('temporalCoverage')},
        {'field_@id': 'dataset:spatialCoverage', 'value': metadata_json.get('spatialCoverage')},
        {'field_@id': 'dataset:description', 'value': metadata_json.get('description')},
        {'field_@id': 'dataset:citeAs', 'value': metadata_json.get('citeAs')}
    ]
    dataframes['dataset_summary'] = pd.DataFrame(dataset_summary_rows)
    print('Created fallback DataFrame: dataset_summary')

for df_name, df in dataframes.items():
    print(f'\nDataFrame: {df_name}')
    print('Columns:', df.columns.tolist())
    display(df.head())

We can also build an analysis-ready table from selected FAIR metadata sections that contain lists, such as `dataUseCases`, `dataBiases`, and `dataLimitations`. This gives us a useful way to explore the dataset contents even when tabular record sets are not provided.

In [ ]:
analysis_rows = []

for item in metadata_json.get('dataUseCases', []) or []:
    analysis_rows.append({'section_@id': 'dataset:dataUseCases', 'text': item, 'text_length': len(item)})

for item in metadata_json.get('dataBiases', []) or []:
    analysis_rows.append({'section_@id': 'dataset:dataBiases', 'text': item, 'text_length': len(item)})

for item in metadata_json.get('dataLimitations', []) or []:
    analysis_rows.append({'section_@id': 'dataset:dataLimitations', 'text': item, 'text_length': len(item)})

for item in metadata_json.get('dataPreprocessingProtocol', []) or []:
    analysis_rows.append({'section_@id': 'dataset:dataPreprocessingProtocol', 'text': item, 'text_length': len(item)})

analysis_df = pd.DataFrame(analysis_rows)
dataframes['dataset_text_sections'] = analysis_df

print('Constructed text analysis table from metadata sections.')
display(analysis_df.head(10))

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps to the extracted data. Since this dataset currently exposes rich descriptive metadata but may not include tabular record sets in the Croissant package, this section demonstrates EDA on the structured metadata-derived table.

We will:
- select a numeric field,
- filter longer text entries,
- normalize the numeric values,
- group by a categorical field referenced by an `@id`-style label.

In [ ]:
record_set_id = 'dataset_text_sections'
numeric_field = 'text_length'
group_field = 'section_@id'
threshold = 120

eda_df = dataframes[record_set_id].copy()

filtered_df = eda_df[eda_df[numeric_field] > threshold].copy()
print(f'Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows')
display(filtered_df.head())

if not filtered_df.empty and filtered_df[numeric_field].std() != 0:
    filtered_df[f'{numeric_field}_normalized'] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
else:
    filtered_df[f'{numeric_field}_normalized'] = 0.0

print(f'Normalized {numeric_field} for filtered records:')
display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].agg(['count', 'mean', 'median', 'max']).reset_index()
    print(f'Grouped data by {group_field}:')
    display(grouped_df)
else:
    print(f'Grouping field {group_field} not found in DataFrame columns.')

In [ ]:
# Additional metadata-focused exploration
dataset_profile = pd.DataFrame([
    {'metric_@id': 'dataset:author_count', 'value': len(metadata_json.get('author', []) or [])},
    {'metric_@id': 'dataset:citation_count', 'value': len(metadata_json.get('citation', []) or [])},
    {'metric_@id': 'dataset:distribution_count', 'value': len(metadata_json.get('distribution', []) or [])},
    {'metric_@id': 'dataset:recordSet_count', 'value': len(metadata_json.get('recordSet', []) or [])},
    {'metric_@id': 'dataset:visualization_count', 'value': len(metadata_json.get('visualization', []) or [])},
    {'metric_@id': 'dataset:keyword_count', 'value': len(metadata_json.get('keywords', []) or [])},
    {'metric_@id': 'dataset:dataBiases_count', 'value': len(metadata_json.get('dataBiases', []) or [])},
    {'metric_@id': 'dataset:dataUseCases_count', 'value': len(metadata_json.get('dataUseCases', []) or [])},
    {'metric_@id': 'dataset:dataLimitations_count', 'value': len(metadata_json.get('dataLimitations', []) or [])},
    {'metric_@id': 'dataset:dataPreprocessingProtocol_count', 'value': len(metadata_json.get('dataPreprocessingProtocol', []) or [])}
])

dataframes['dataset_profile'] = dataset_profile
display(dataset_profile)

## 5. Visualization
Visualize distributions and grouped summaries derived from the metadata-based analysis tables.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=analysis_df, x='text_length', bins=12, kde=True)
plt.title('Distribution of text lengths across metadata sections')
plt.xlabel('Text length (characters)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
section_summary = analysis_df.groupby('section_@id', as_index=False)['text_length'].mean().sort_values('text_length', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=section_summary, x='section_@id', y='text_length', palette='viridis')
plt.title('Average text length by metadata section @id')
plt.xlabel('Section @id')
plt.ylabel('Average text length')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=dataset_profile, x='metric_@id', y='value', palette='magma')
plt.title('High-level dataset profile metrics')
plt.xlabel('Metric @id')
plt.ylabel('Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Conclusion
This notebook loaded the FAIR^2 Croissant package for the bat occurrence dataset and explored its metadata using `mlcroissant`.

### Key observations
- The dataset metadata includes rich descriptive FAIR information such as collection methods, biases, limitations, preprocessing steps, authorship, citations, and usage guidance.
- In the currently exposed Croissant metadata, no tabular `recordSet` entries were available, so direct row-level occurrence extraction through `dataset.records(record_set=<@id>)` could not be demonstrated on a real record set.
- To keep the notebook useful and reproducible, we built analysis tables from metadata sections using explicit `@id`-style identifiers.
- The metadata indicates that the resource supports biodiversity, macroecological, and conservation analysis, while also documenting important limitations such as uneven sampling and incomplete metadata.

If a future version of the Croissant package includes record sets and field definitions for the occurrence table itself, the same notebook structure can be reused by swapping in the discovered record set `@id` values and loading those records directly into pandas DataFrames.